# 2.5D ResNet-50 with 5-Fold Cross-Validation in Google Colab

This notebook trains a **2.5D ResNet-50** using parenchyma CT images from three adjacent axial slices. For every central slice `z`, the model receives `[z-1, z, z+1]` as its three input channels. Missing neighbours are replaced by the central slice when `BOUNDARY_MODE='replicate_center'`.

The dataset archive is read from Google Drive, copied to Colab local storage, extracted, and validated before training. Cross-validation checkpoints, metrics, configurations, holdout predictions, Grad-CAM maps, and LRP maps are saved back to Google Drive.

Before running this notebook, push the latest `cv_2_5d_resnet50` code to the configured repository branch and select **Runtime > Change runtime type > GPU**. Run the cells in order.

> Each `EXPERIMENT_ID` may only be used for one new training run because the training program intentionally refuses to overwrite an existing output directory.

## Dataset archive already uploaded to Google Drive

This notebook expects the following archive by default:

`MyDrive/mask-guided-lung-nodule-xai/a0d90f9e-3dd4-4de0-98af-12858696f613_cv_resnet50_parenchyma.tar.gz`

The archive name contains `cv_resnet50`, but its contents are reusable for 2.5D training. It contains the cross-validation metadata, parenchyma CT slices, and ground-truth masks—not trained 2D model checkpoints. The directory structure inside the archive must start with `000_dataset_v2`. Do not flatten the archive during extraction.

## 1. Check the GPU

In [ ]:
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB')

## 2. Mount Google Drive

The dataset archive is loaded from Google Drive. Experiment outputs are written directly to Drive so they survive a Colab runtime reset.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Configure the complete experiment

This is the **only parameter cell**. It exposes repository, archive, extraction, dataset, 2.5D window, model, optimizer, augmentation, DataLoader, training, testing, and XAI settings. Edit this cell before continuing.

In [ ]:
from pathlib import Path

# -----------------------------------------------------------------
# Repository
# -----------------------------------------------------------------
REPOSITORY_URL = 'https://github.com/FillipusAditya/mask-guided-lung-nodule-xai.git'
REPOSITORY_BRANCH = 'refactor/002-segmentation'
PROJECT_ROOT = Path('/content/mask-guided-lung-nodule-xai')

# -----------------------------------------------------------------
# Execution switches
# -----------------------------------------------------------------
REQUIRE_GPU = True
INSTALL_REQUIREMENTS = True
RUN_TRAINING = True
RUN_TEST_AFTER_TRAINING = False

# -----------------------------------------------------------------
# Experiment identity
# -----------------------------------------------------------------
EXPERIMENT_ID = 'a0d90f9e-3dd4-4de0-98af-12858696f613'
EXPERIMENT_COMPONENT = 'classification/cv_2_5d_resnet50'

# -----------------------------------------------------------------
# Google Drive archive and local extraction
# -----------------------------------------------------------------
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/mask-guided-lung-nodule-xai')
ARCHIVE_NAME = 'a0d90f9e-3dd4-4de0-98af-12858696f613_cv_resnet50_parenchyma.tar.gz'
DRIVE_ARCHIVE_PATH = DRIVE_PROJECT_ROOT / ARCHIVE_NAME
LOCAL_ARCHIVE_PATH = Path('/content') / ARCHIVE_NAME
EXTRACTION_ROOT = Path('/content/classification_training_data')
DATASET_ROOT = EXTRACTION_ROOT / '000_dataset_v2/_segmentation_dataset'
COPY_ARCHIVE_TO_LOCAL = True
FORCE_EXTRACT = False
VERIFY_ARCHIVE_SHA256 = False
EXPECTED_ARCHIVE_SHA256 = None  # Optional lowercase SHA-256 string.

# -----------------------------------------------------------------
# Persistent outputs
# -----------------------------------------------------------------
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_ROOT / 'experiment_results'
DRIVE_OUTPUT_DIR = DRIVE_OUTPUT_ROOT / EXPERIMENT_ID / EXPERIMENT_COMPONENT

# -----------------------------------------------------------------
# Input data and true 2.5D window
# -----------------------------------------------------------------
METADATA_FILENAME = '004_classification_cv_5fold_seed42.csv'
CT_PATH_COLUMN = 'ct_parenchyma_path'
INPUT_HEIGHT = 224
INPUT_WIDTH = 224
SLICE_OFFSETS = [-1, 0, 1]
BOUNDARY_MODE = 'replicate_center'  # 'replicate_center' or 'valid_only'
CLASS_TO_IDX = {'benign': 0, 'malignant': 1}
NORMALIZATION_MEAN = [0.485, 0.456, 0.406]
NORMALIZATION_STD = [0.229, 0.224, 0.225]

# Cross-validation metadata contract
NUM_FOLDS = 5
DEVELOPMENT_ROLE = 'development'
HOLDOUT_ROLE = 'holdout_test'
HOLDOUT_FOLD = -1
GROUP_COLUMN = 'cv_group_id'
NODULE_COLUMN = 'cv_nodule_id'
FOLD_COLUMN = 'cv_fold'
ROLE_COLUMN = 'cv_role'

# -----------------------------------------------------------------
# ResNet-50 model
# -----------------------------------------------------------------
MODEL_ARCHITECTURE = '2.5D ResNet50'
PRETRAINED_WEIGHTS = 'DEFAULT'  # 'DEFAULT', 'IMAGENET1K_V2', or 'NONE'
TRAINING_STRATEGY = 'full_fine_tuning'
CLASSIFIER_DROPOUT = 0.3
NUM_CLASSES = 2

# -----------------------------------------------------------------
# Training
# -----------------------------------------------------------------
NUM_EPOCHS = 100
BATCH_SIZE = 32  # Reduce if GPU memory is insufficient.
LEARNING_RATE = 1e-3
SEED = 42
TRANSFORM_SEED = 42
CLASSIFICATION_THRESHOLD = 0.5
DEVICE = 'auto'  # 'auto', 'cuda', or 'cpu'

# SGD optimizer
OPTIMIZER_NAME = 'SGD'
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4
NESTEROV = False

# Early stopping
EARLY_STOPPING_MONITOR = 'val_loss'
EARLY_STOPPING_MODE = 'min'
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 0.0

# DataLoader
NUM_WORKERS = 2
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 2
PIN_MEMORY = True
TRAIN_SHUFFLE = True
VAL_SHUFFLE = False
TRAIN_DROP_LAST = False
VAL_DROP_LAST = False

# Training augmentation
HORIZONTAL_FLIP_PROBABILITY = 0.5
ROTATION_LIMIT = 15
ROTATION_PROBABILITY = 0.5
BRIGHTNESS_LIMIT = 0.1
CONTRAST_LIMIT = 0.1
BRIGHTNESS_CONTRAST_PROBABILITY = 0.3
NOISE_STD_RANGE = [0.01, 0.03]
NOISE_PROBABILITY = 0.2

# Nodule-level aggregation
NODULE_AGGREGATION_METHOD = 'mean_probability'

# Explainability
GRADCAM_LAYER = 'layer4[-1]'
LRP_RULE = 'EpsilonPlusFlat'
XAI_TARGET = 'predicted_class'  # 'predicted_class' or 'true_class'

# -----------------------------------------------------------------
# Holdout testing and visualization
# -----------------------------------------------------------------
MAX_TEST_SAMPLES = None  # Use 8 for a quick smoke test.
TEST_BATCH_SIZE = 1
TEST_NUM_WORKERS = 0
TEST_DEVICE = 'auto'
SKIP_XAI = False
FORCE_TEST_INFERENCE = False
VISUALIZATION_DPI = 120
VISUALIZATION_ROWS_PER_FIGURE = 12

print(f'Archive: {DRIVE_ARCHIVE_PATH}')
print(f'Dataset: {DATASET_ROOT}')
print(f'Output: {DRIVE_OUTPUT_DIR}')

## 4. Clone or update the repository

The repository is cloned to Colab local storage for faster imports. Existing clones are updated to the configured branch.

In [ ]:
import subprocess

if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError('GPU is unavailable. Enable a GPU in Runtime settings.')

if (PROJECT_ROOT / '.git').is_dir():
    print('Updating the existing repository...')
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', REPOSITORY_BRANCH], check=True)
    subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', REPOSITORY_BRANCH],
        check=True,
    )
else:
    print('Cloning the repository...')
    subprocess.run(
        ['git', 'clone', '--branch', REPOSITORY_BRANCH, '--single-branch',
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )

required_program_files = [
    PROJECT_ROOT / '003_classification/cv_2_5d_resnet50/train.py',
    PROJECT_ROOT / '003_classification/cv_2_5d_resnet50/test.py',
    PROJECT_ROOT / '003_classification/cv_2_5d_resnet50/dataset.py',
]
for program_file in required_program_files:
    if not program_file.is_file():
        raise FileNotFoundError(f'Required program file not found: {program_file}')

print(f'Repository is ready: {PROJECT_ROOT}')

## 5. Install the required packages

The CUDA-compatible PyTorch and torchvision versions supplied by Colab are retained. The remaining packages are installed from the program requirements file.

In [ ]:
import sys

requirements_path = PROJECT_ROOT / '003_classification/cv_2_5d_resnet50/requirements.txt'
if INSTALL_REQUIREMENTS:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)],
        check=True,
    )
print('All dependencies are ready.')

## 6. Copy and extract the dataset archive

The archive is copied from Google Drive to Colab local storage by default, which makes extraction and training-data access faster. A completion marker prevents unnecessary repeated extraction. Set `FORCE_EXTRACT=True` only when the local extracted copy must be rebuilt.

In [ ]:
import hashlib
import json
import shutil
import tarfile

from tqdm.auto import tqdm


def copy_file_with_progress(source, destination, chunk_size=8 * 1024 * 1024):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = destination.with_suffix(destination.suffix + '.part')
    total_bytes = source.stat().st_size
    with source.open('rb') as input_file, temporary_path.open('wb') as output_file:
        with tqdm(total=total_bytes, desc='Copying archive', unit='B', unit_scale=True) as bar:
            while True:
                chunk = input_file.read(chunk_size)
                if not chunk:
                    break
                output_file.write(chunk)
                bar.update(len(chunk))
    temporary_path.replace(destination)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as file:
        with tqdm(total=path.stat().st_size, desc='SHA-256', unit='B', unit_scale=True) as bar:
            while True:
                chunk = file.read(chunk_size)
                if not chunk:
                    break
                digest.update(chunk)
                bar.update(len(chunk))
    return digest.hexdigest()


if not DRIVE_ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f'Dataset archive not found: {DRIVE_ARCHIVE_PATH}')

if COPY_ARCHIVE_TO_LOCAL:
    archive_path = LOCAL_ARCHIVE_PATH
    same_size = (
        archive_path.is_file()
        and archive_path.stat().st_size == DRIVE_ARCHIVE_PATH.stat().st_size
    )
    if same_size:
        print(f'Using existing local archive: {archive_path}')
    else:
        copy_file_with_progress(DRIVE_ARCHIVE_PATH, archive_path)
else:
    archive_path = DRIVE_ARCHIVE_PATH

if VERIFY_ARCHIVE_SHA256 or EXPECTED_ARCHIVE_SHA256:
    actual_sha256 = sha256_file(archive_path)
    print(f'Archive SHA-256: {actual_sha256}')
    if EXPECTED_ARCHIVE_SHA256 and actual_sha256 != EXPECTED_ARCHIVE_SHA256.lower():
        raise ValueError('Archive SHA-256 does not match EXPECTED_ARCHIVE_SHA256.')

extraction_marker = EXTRACTION_ROOT / '.extraction_complete.json'
archive_identity = {
    'archive_name': ARCHIVE_NAME,
    'archive_size': archive_path.stat().st_size,
}
marker_matches = False
if extraction_marker.is_file():
    marker_matches = json.loads(extraction_marker.read_text()) == archive_identity

if FORCE_EXTRACT and EXTRACTION_ROOT.exists():
    if EXTRACTION_ROOT.parent != Path('/content') or EXTRACTION_ROOT == Path('/content'):
        raise ValueError(f'Unsafe extraction root for deletion: {EXTRACTION_ROOT}')
    shutil.rmtree(EXTRACTION_ROOT)
    marker_matches = False
elif extraction_marker.is_file() and not marker_matches:
    raise RuntimeError('Extraction marker does not match this archive. Set FORCE_EXTRACT=True.')

if marker_matches:
    print(f'Using extracted dataset: {EXTRACTION_ROOT}')
else:
    EXTRACTION_ROOT.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, mode='r:gz') as archive:
        members = archive.getmembers()
        if not any(member.name.startswith('000_dataset_v2/') for member in members):
            raise ValueError('Archive must contain the top-level 000_dataset_v2 directory.')
        for member in tqdm(members, desc='Extracting dataset', unit='file'):
            archive.extract(member, path=EXTRACTION_ROOT, filter='data')
    extraction_marker.write_text(json.dumps(archive_identity, indent=2) + '\n')

print(f'Dataset is ready: {DATASET_ROOT}')

## 7. Validate the dataset and 2.5D neighbourhoods

This cell validates metadata columns, class labels, five-fold assignments, patient/nodule isolation, slice-index parsing, duplicate slices, parenchyma paths, and masks. It also reports how many windows require boundary replication.

In [ ]:
import csv
import re

metadata_path = DATASET_ROOT / METADATA_FILENAME
required_directories = [
    EXTRACTION_ROOT / '000_dataset_v2/_lidc/007_segmentation_dataset_npy/ct_parenchyma',
    EXTRACTION_ROOT / '000_dataset_v2/_lndb/007_segmentation_dataset_npy/ct_parenchyma',
    EXTRACTION_ROOT / '000_dataset_v2/_lidc/007_segmentation_dataset_npy/mask',
    EXTRACTION_ROOT / '000_dataset_v2/_lndb/007_segmentation_dataset_npy/mask',
]
required_columns = {
    'dataset', 'patient_id', 'filename', CT_PATH_COLUMN, 'mask_path',
    'label', GROUP_COLUMN, NODULE_COLUMN, ROLE_COLUMN, FOLD_COLUMN,
}
slice_pattern = re.compile(r'_slice_(\d+)\.npy$')

if not metadata_path.is_file():
    raise FileNotFoundError(f'Metadata not found: {metadata_path}')
for directory in required_directories:
    if not directory.is_dir():
        raise FileNotFoundError(f'Required directory not found: {directory}')

with metadata_path.open('r', encoding='utf-8', newline='') as file:
    reader = csv.DictReader(file)
    column_names = set(reader.fieldnames or [])
    rows = list(reader)

missing_columns = required_columns - column_names
if missing_columns:
    raise ValueError(f'Metadata columns are incomplete: {sorted(missing_columns)}')
if not rows:
    raise ValueError('Metadata must not be empty.')

missing_files = []
nodule_slices = {}
nodule_labels = {}
nodule_groups = {}
for row in tqdm(rows, desc='Validating samples', unit='sample'):
    match = slice_pattern.search(row['filename'])
    if match is None:
        raise ValueError(f"Cannot parse slice index: {row['filename']}")
    slice_index = int(match.group(1))
    nodule_id = row[NODULE_COLUMN]
    if slice_index in nodule_slices.setdefault(nodule_id, set()):
        raise ValueError(f'Duplicate slice {slice_index} in nodule {nodule_id}.')
    nodule_slices[nodule_id].add(slice_index)
    nodule_labels.setdefault(nodule_id, set()).add(row['label'].strip().lower())
    nodule_groups.setdefault(nodule_id, set()).add(row[GROUP_COLUMN])
    for path_column in (CT_PATH_COLUMN, 'mask_path'):
        path = DATASET_ROOT / row[path_column]
        if not path.is_file():
            missing_files.append(path)

if missing_files:
    examples = '\n'.join(str(path) for path in missing_files[:5])
    raise FileNotFoundError(f'{len(missing_files)} files are missing.\n{examples}')
if any(len(labels) != 1 for labels in nodule_labels.values()):
    raise ValueError('At least one nodule has inconsistent labels.')
if any(len(groups) != 1 for groups in nodule_groups.values()):
    raise ValueError('At least one nodule belongs to multiple patient groups.')

development_rows = [r for r in rows if r[ROLE_COLUMN].strip().lower() == DEVELOPMENT_ROLE]
holdout_rows = [r for r in rows if r[ROLE_COLUMN].strip().lower() == HOLDOUT_ROLE]
development_folds = {int(row[FOLD_COLUMN]) for row in development_rows}
if development_folds != set(range(NUM_FOLDS)):
    raise ValueError(f'Unexpected development folds: {sorted(development_folds)}')
development_patients = {row[GROUP_COLUMN] for row in development_rows}
holdout_patients = {row[GROUP_COLUMN] for row in holdout_rows}
if development_patients & holdout_patients:
    raise RuntimeError('Patient leakage exists between development and holdout.')

label_counts = {}
for row in rows:
    label = row['label'].strip().lower()
    label_counts[label] = label_counts.get(label, 0) + 1
if set(label_counts) != set(CLASS_TO_IDX):
    raise ValueError(f'Unexpected classes: {sorted(label_counts)}')

replicated_windows = 0
for indices in nodule_slices.values():
    for center in indices:
        if any(center + offset not in indices for offset in SLICE_OFFSETS):
            replicated_windows += 1

nodule_sizes = [len(indices) for indices in nodule_slices.values()]
print(f'Number of slices/windows: {len(rows):,}')
print(f'Number of nodules: {len(nodule_slices):,}')
print(f'Class distribution: {label_counts}')
print(f'Development folds: {sorted(development_folds)}')
print(f'Nodule size range: {min(nodule_sizes)} to {max(nodule_sizes)} slices')
print(f'Windows requiring centre replication: {replicated_windows:,}')
print('All parenchyma images, masks, and 2.5D metadata relationships are valid.')

## 8. Create and save the JSON configuration

The configuration is built exclusively from the parameter cell. One copy is written into the local repository for training and another is saved persistently in Google Drive.

In [ ]:
if NUM_FOLDS != 5:
    raise ValueError('This metadata requires NUM_FOLDS = 5.')
if SLICE_OFFSETS != [-1, 0, 1]:
    raise ValueError('The pretrained ResNet-50 implementation requires [-1, 0, 1].')
if BOUNDARY_MODE not in {'replicate_center', 'valid_only'}:
    raise ValueError('Invalid BOUNDARY_MODE.')
if BATCH_SIZE < 1 or NUM_EPOCHS < 1 or LEARNING_RATE <= 0:
    raise ValueError('Batch size, epochs, and learning rate must be positive.')
if NUM_WORKERS < 0 or TEST_NUM_WORKERS < 0:
    raise ValueError('DataLoader worker counts cannot be negative.')
if not 0.0 <= CLASSIFICATION_THRESHOLD <= 1.0:
    raise ValueError('CLASSIFICATION_THRESHOLD must be between zero and one.')
if DEVICE not in {'auto', 'cpu', 'cuda'} or TEST_DEVICE not in {'auto', 'cpu', 'cuda'}:
    raise ValueError('DEVICE and TEST_DEVICE must be auto, cpu, or cuda.')
if XAI_TARGET not in {'predicted_class', 'true_class'}:
    raise ValueError('XAI_TARGET must be predicted_class or true_class.')

config = {
    'experiment': {'id': EXPERIMENT_ID, 'component': EXPERIMENT_COMPONENT},
    'output': {'root_directory': str(DRIVE_OUTPUT_ROOT)},
    'data': {
        'dataset_root': str(DATASET_ROOT), 'metadata_path': str(metadata_path),
        'ct_path_column': CT_PATH_COLUMN, 'input_height': INPUT_HEIGHT,
        'input_width': INPUT_WIDTH, 'slice_offsets': SLICE_OFFSETS,
        'boundary_mode': BOUNDARY_MODE, 'class_to_idx': CLASS_TO_IDX,
        'normalization_mean': NORMALIZATION_MEAN, 'normalization_std': NORMALIZATION_STD,
    },
    'cross_validation': {
        'num_folds': NUM_FOLDS, 'development_role': DEVELOPMENT_ROLE,
        'holdout_role': HOLDOUT_ROLE, 'holdout_fold': HOLDOUT_FOLD,
        'group_column': GROUP_COLUMN, 'nodule_column': NODULE_COLUMN,
        'fold_column': FOLD_COLUMN, 'role_column': ROLE_COLUMN,
    },
    'model': {
        'architecture': MODEL_ARCHITECTURE, 'pretrained_weights': PRETRAINED_WEIGHTS,
        'training_strategy': TRAINING_STRATEGY,
        'classifier_dropout': CLASSIFIER_DROPOUT, 'num_classes': NUM_CLASSES,
    },
    'training': {
        'num_epochs': NUM_EPOCHS, 'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE, 'seed': SEED,
        'transform_seed': TRANSFORM_SEED,
        'classification_threshold': CLASSIFICATION_THRESHOLD, 'device': DEVICE,
    },
    'optimizer': {
        'name': OPTIMIZER_NAME, 'momentum': MOMENTUM,
        'weight_decay': WEIGHT_DECAY, 'nesterov': NESTEROV,
    },
    'dataloader': {
        'num_workers': NUM_WORKERS, 'persistent_workers': PERSISTENT_WORKERS,
        'prefetch_factor': PREFETCH_FACTOR, 'pin_memory': PIN_MEMORY,
        'train_shuffle': TRAIN_SHUFFLE, 'val_shuffle': VAL_SHUFFLE,
        'train_drop_last': TRAIN_DROP_LAST, 'val_drop_last': VAL_DROP_LAST,
    },
    'early_stopping': {
        'monitor': EARLY_STOPPING_MONITOR, 'mode': EARLY_STOPPING_MODE,
        'patience': EARLY_STOPPING_PATIENCE,
        'min_delta': EARLY_STOPPING_MIN_DELTA,
    },
    'aggregation': {'nodule_method': NODULE_AGGREGATION_METHOD},
    'augmentation': {
        'horizontal_flip_probability': HORIZONTAL_FLIP_PROBABILITY,
        'rotation_limit': ROTATION_LIMIT, 'rotation_probability': ROTATION_PROBABILITY,
        'brightness_limit': BRIGHTNESS_LIMIT, 'contrast_limit': CONTRAST_LIMIT,
        'brightness_contrast_probability': BRIGHTNESS_CONTRAST_PROBABILITY,
        'noise_std_range': NOISE_STD_RANGE, 'noise_probability': NOISE_PROBABILITY,
    },
    'xai': {
        'gradcam_layer': GRADCAM_LAYER, 'lrp_rule': LRP_RULE, 'target': XAI_TARGET,
    },
}


def save_json(data, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(data, indent=2) + '\n', encoding='utf-8')


config_path = PROJECT_ROOT / '003_classification/configs/cv_2_5d_resnet50.json'
drive_config_path = (
    DRIVE_OUTPUT_ROOT / 'saved_configs' / EXPERIMENT_ID / 'cv_2_5d_resnet50.json'
)
save_json(config, config_path)
save_json(config, drive_config_path)

print(json.dumps(config, indent=2))
print(f'Local configuration: {config_path}')
print(f'Drive configuration: {drive_config_path}')

## 9. Run the 2.5D preflight check

The preflight check uses the actual training configuration. It creates fold 0 loaders, selects a non-boundary validation window when available, confirms the three slice indices, downloads the requested ImageNet weights, and verifies the model input/output shapes.

In [ ]:
import importlib

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
importlib.invalidate_caches()
module_name = '003_classification.cv_2_5d_resnet50.train'
if module_name in sys.modules:
    train_module = importlib.reload(sys.modules[module_name])
else:
    train_module = importlib.import_module(module_name)

effective_config = train_module.load_config(config_path, DRIVE_OUTPUT_DIR)
train_module.validate_configuration(effective_config)
train_module.validate_metadata(effective_config)
train_loader, val_loader = train_module.make_fold_loaders(effective_config, fold=0)
validation_dataset = val_loader.dataset
sample_position = next(
    (i for i, window in enumerate(validation_dataset.windows)
     if not window['boundary_replicated']),
    0,
)
sample_image, sample_label, sample_index = validation_dataset[sample_position]
sample_window = validation_dataset.windows[sample_position]
device = train_module.select_device(DEVICE)
model = train_module.build_model(effective_config, load_pretrained=True).to(device).eval()
with torch.no_grad():
    sample_output = model(sample_image.unsqueeze(0).to(device))

if tuple(sample_image.shape) != (3, INPUT_HEIGHT, INPUT_WIDTH):
    raise RuntimeError(f'Unexpected input shape: {tuple(sample_image.shape)}')
if tuple(sample_output.shape) != (1, NUM_CLASSES):
    raise RuntimeError(f'Unexpected output shape: {tuple(sample_output.shape)}')

print(f'Fold 0 training windows: {len(train_loader.dataset):,}')
print(f'Fold 0 validation windows: {len(validation_dataset):,}')
print(f'Input slice indices: {sample_window["input_slice_indices"]}')
print(f'Input filenames: {sample_window["input_filenames"]}')
print(f'Boundary replicated: {sample_window["boundary_replicated"]}')
print(f'Input shape: {tuple(sample_image.shape)}')
print(f'Output shape: {tuple(sample_output.shape)}')
print(f'Sample label: {int(sample_label)}; dataset index: {int(sample_index)}')

del model, train_loader, val_loader
torch.cuda.empty_cache()

## 10. Start five-fold training

Each fold starts from a fresh pretrained ResNet-50 and optimizer. The holdout partition is never used during training. Best checkpoints are selected by validation loss; both window-level and nodule-level validation metrics are saved.

In [ ]:
if RUN_TRAINING:
    if DRIVE_OUTPUT_DIR.exists():
        raise FileExistsError(
            f'Output already exists: {DRIVE_OUTPUT_DIR}\n'
            'Change EXPERIMENT_ID to start a new training run.'
        )
    train_command = [
        sys.executable, '-u', '-m', '003_classification.cv_2_5d_resnet50.train',
        '--config', str(config_path), '--output-dir', str(DRIVE_OUTPUT_DIR),
    ]
    print('Starting 2.5D ResNet-50 training...', flush=True)
    print(f'Command: {" ".join(train_command)}', flush=True)
    subprocess.run(train_command, cwd=PROJECT_ROOT, check=True)
else:
    print('Training was skipped because RUN_TRAINING=False.')

## 11. View cross-validation results

This section displays fold summaries and the combined window/nodule metric figure, then lists the principal artifacts.

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary_path = DRIVE_OUTPUT_DIR / 'cv_summary.csv'
summary_json_path = DRIVE_OUTPUT_DIR / 'cv_summary.json'
plot_path = DRIVE_OUTPUT_DIR / 'figures/cv_fold_metrics.png'
config_snapshot_path = DRIVE_OUTPUT_DIR / 'cv_2_5d_resnet50.json'
oof_window_path = DRIVE_OUTPUT_DIR / 'out_of_fold_predictions.csv'
oof_nodule_path = DRIVE_OUTPUT_DIR / 'out_of_fold_nodule_predictions.csv'

if not summary_path.is_file():
    raise FileNotFoundError(f'Training summary not found: {summary_path}')
display(pd.read_csv(summary_path))
if plot_path.is_file():
    display(Image(filename=str(plot_path), width=850))

print(f'Output directory: {DRIVE_OUTPUT_DIR}')
print(f'Configuration snapshot: {config_snapshot_path}')
print(f'OOF window predictions: {oof_window_path}')
print(f'OOF nodule predictions: {oof_nodule_path}')
for fold in range(NUM_FOLDS):
    print(f'Fold {fold} best model: {DRIVE_OUTPUT_DIR / f"fold_{fold}/best_model.pth"}')

## 12. Run holdout testing and XAI (optional)

Set `RUN_TEST_AFTER_TRAINING=True` in the parameter cell to evaluate the five-fold ensemble. Set `SKIP_XAI=True` for metrics-only evaluation. Full Grad-CAM and LRP generation is computationally expensive; use `MAX_TEST_SAMPLES=8` for an initial smoke test.

Standard Grad-CAM produces one map for the complete three-slice window and is displayed on the central slice. LRP preserves three input channels and additionally creates averaged relevance maps for each physical slice.

In [ ]:
if RUN_TEST_AFTER_TRAINING:
    test_command = [
        sys.executable, '-u', '-m', '003_classification.cv_2_5d_resnet50.test',
        str(DRIVE_OUTPUT_DIR), '--batch-size', str(TEST_BATCH_SIZE),
        '--num-workers', str(TEST_NUM_WORKERS), '--device', TEST_DEVICE,
        '--dpi', str(VISUALIZATION_DPI),
        '--rows-per-figure', str(VISUALIZATION_ROWS_PER_FIGURE),
    ]
    if MAX_TEST_SAMPLES is not None:
        test_command.extend(['--max-samples', str(MAX_TEST_SAMPLES)])
    if SKIP_XAI:
        test_command.append('--skip-xai')
    if FORCE_TEST_INFERENCE:
        test_command.append('--force-inference')
    print(f'Command: {" ".join(test_command)}', flush=True)
    subprocess.run(test_command, cwd=PROJECT_ROOT, check=True)
    print(f'Test output: {DRIVE_OUTPUT_DIR / "test"}')
else:
    print('Testing was skipped because RUN_TEST_AFTER_TRAINING=False.')

## 13. Inspect holdout metrics and XAI outputs

Run this cell after holdout testing. It displays window-level and nodule-level results, prediction examples, and one generated XAI figure when available.

In [ ]:
test_dir = DRIVE_OUTPUT_DIR / 'test'
test_results_path = test_dir / 'test_results.json'
window_predictions_path = test_dir / 'test_predictions.csv'
nodule_predictions_path = test_dir / 'test_nodule_predictions.csv'

if test_results_path.is_file():
    test_results = json.loads(test_results_path.read_text(encoding='utf-8'))
    print(json.dumps(test_results, indent=2))
    if window_predictions_path.is_file():
        print('\nWindow-level prediction examples:')
        display(pd.read_csv(window_predictions_path).head())
    if nodule_predictions_path.is_file():
        print('\nNodule-level prediction examples:')
        display(pd.read_csv(nodule_predictions_path).head())
    visualization_files = sorted((test_dir / 'visualization').glob('*.png'))
    if visualization_files:
        display(Image(filename=str(visualization_files[0]), width=1200))
else:
    print(f'Test results are not available: {test_results_path}')

## Run a new experiment

The training program creates a new output directory and does not overwrite or automatically resume an existing experiment. To start another run, change `EXPERIMENT_ID` in the parameter cell, optionally adjust other settings, and rerun from Section 8. The extracted dataset can be reused as long as the Colab runtime remains active.

If the runtime restarts, rerun all sections so Google Drive is mounted, the repository is restored, and the archive is extracted to local storage again.